In [2]:
# llm model: skt/A.X-4.0
# python 3.12.7
import torch
from typing import List, TypedDict
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langgraph.graph import END, StateGraph

# ==========================================
# 1. 상태 정의 (Graph State)
# ==========================================
class GraphState(TypedDict):
    question: str
    context: List[str]
    answer: str

# ==========================================
# 2. 모델 및 임베딩 로드 (Mac M시리즈 최적화)
# ==========================================
print("모델을 로드하는 중입니다. (시간이 꽤 걸릴 수 있습니다...)")

# Apple Silicon(M1/M2/M3) 가속 사용 설정
device = "mps" if torch.backends.mps.is_available() else "cpu"

# SKT A.X-4.0 모델 로드
model_id = "skt/A.X-4.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Mac의 메모리 한계를 고려해 float16으로 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16, 
    device_map=device
)

# 텍스트 생성 파이프라인 구축
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,      # 답변 최대 길이
    temperature=0.1,         # RAG는 사실 기반이어야 하므로 온도를 낮춤
    do_sample=True,
    return_full_text=False   # 프롬프트 제외하고 답변만 반환
)

# LangChain용 LLM으로 래핑
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# 한국어 임베딩 모델 로드
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# ==========================================
# 3. 예시 데이터 및 리트리버 세팅
# ==========================================
sample_texts = [
    "SKT의 A.X는 인공지능 언어모델입니다.",
    "LangGraph는 복잡한 에이전트 워크플로우를 만들기 위한 프레임워크입니다.",
    "루키 에이전트(Rookie Agent) 프로젝트는 성공적으로 진행될 것입니다."
]
docs = [Document(page_content=text) for text in sample_texts]

# 벡터 데이터베이스(Chroma) 생성
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2}) # 관련 문서 2개 검색

# ==========================================
# 4. LangGraph 노드(Node) 정의
# ==========================================
def retrieve_node(state: GraphState):
    """문서를 검색하는 노드"""
    print("---[노드: 검색 중]---")
    question = state["question"]
    retrieved_docs = retriever.invoke(question)
    context_list = [doc.page_content for doc in retrieved_docs]
    return {"context": context_list}

def generate_node(state: GraphState):
    """답변을 생성하는 노드"""
    print("---[노드: 답변 생성 중]---")
    question = state["question"]
    context_str = "\n".join(state["context"])
    
    # RAG 프롬프트
    prompt = (
        f"당신은 친절하고 정확한 AI 어시스턴트입니다. "
        f"다음 제공된 문맥(Context)을 바탕으로 질문(Question)에 답하세요.\n\n"
        f"[Context]\n{context_str}\n\n"
        f"[Question]\n{question}\n\n"
        f"답변:"
    )
    
    response = llm.invoke(prompt)
    return {"answer": response}

# ==========================================
# 5. 그래프(Workflow) 구성
# ==========================================
workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

# 흐름 연결: 시작 -> retrieve -> generate -> 끝
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# 앱 컴파일
app = workflow.compile()

# ==========================================
# 6. 실행 테스트
# ==========================================
if __name__ == "__main__":
    test_question = "LangGraph가 뭐야?"
    print(f"질문: {test_question}\n")
    
    inputs = {"question": test_question}
    
    for output in app.stream(inputs):
        for key, value in output.items():
            print(value)
            #pass # 진행 과정에서 상태를 보고 싶다면 print(value)
            
    print("\n최종 답변:")
    print(value["answer"])

모델을 로드하는 중입니다. (시간이 꽤 걸릴 수 있습니다...)


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`